In [ ]:
# ==================== JSON Parser ====================

class JSONParser:
    """
    defining JSON parser
    Supports: object, array, string, number, boolean, null, nested structures
    """
    
    def __init__(self):
        self.text = ""
        self.index = 0
    
    def load(self, filepath):
        """
        Load JSON from a file
        
        Parameters:
            filepath: str - path to the JSON file
        
        Returns:
            Parsed Python object (dict, list, etc.)
        """
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                self.text = f.read()
            self.index = 0
            return self.parse_value()
        except FileNotFoundError:
            print(f"Error: File not found at {filepath}")
            return None
        except Exception as e:
            print(f"Error parsing JSON: {e}")
            return None
    
    def loads(self, json_string):
        """
        Load JSON from a string
        
        Parameters:
            json_string: str - JSON string
        
        Returns:
            Parsed Python object
        """
        self.text = json_string
        self.index = 0
        return self.parse_value()
    
    def skip_whitespace(self):
        """Skip whitespace characters"""
        while self.index < len(self.text) and self.text[self.index] in ' \t\n\r':
            self.index += 1
    
    def parse_value(self):
        """Parse JSON value"""
        self.skip_whitespace()
        
        if self.index >= len(self.text):
            raise ValueError("Unexpected end of JSON")
        
        char = self.text[self.index]
        
        # Determine the type of value
        if char == '{':
            return self.parse_object()
        elif char == '[':
            return self.parse_array()
        elif char == '"':
            return self.parse_string()
        elif char == 't':
            return self.parse_true()
        elif char == 'f':
            return self.parse_false()
        elif char == 'n':
            return self.parse_null()
        elif char == '-' or char.isdigit():
            return self.parse_number()
        else:
            raise ValueError(f"Unexpected character: {char} at position {self.index}")
    
    def parse_object(self):
        """Parse JSON object {}"""
        result = {}
        self.index += 1  # Skip '{'
        self.skip_whitespace()
        
        # Empty object
        if self.index < len(self.text) and self.text[self.index] == '}':
            self.index += 1
            return result
        
        while True:
            self.skip_whitespace()
            
            # Parse key (must be a string)
            if self.text[self.index] != '"':
                raise ValueError(f"Expected string key at position {self.index}")
            
            key = self.parse_string()
            self.skip_whitespace()
            
            # Expect ':'
            if self.text[self.index] != ':':
                raise ValueError(f"Expected ':' at position {self.index}")
            self.index += 1
            
            # Parse value
            value = self.parse_value()
            result[key] = value
            
            self.skip_whitespace()
            
            # Check if end or continue
            if self.text[self.index] == '}':
                self.index += 1
                break
            elif self.text[self.index] == ',':
                self.index += 1
            else:
                raise ValueError(f"Expected ',' or '}}' at position {self.index}")
        
        return result
    
    def parse_array(self):
        """Parse JSON array []"""
        result = []
        self.index += 1  # Skip '['
        self.skip_whitespace()
        
        # Empty array
        if self.index < len(self.text) and self.text[self.index] == ']':
            self.index += 1
            return result
        
        while True:
            # Parse value
            value = self.parse_value()
            result.append(value)
            
            self.skip_whitespace()
            
            # Check if end or continue
            if self.text[self.index] == ']':
                self.index += 1
                break
            elif self.text[self.index] == ',':
                self.index += 1
            else:
                raise ValueError(f"Expected ',' or ']' at position {self.index}")
        
        return result
    
    def parse_string(self):
        """Parse string"""
        self.index += 1  # Skip starting '"'
        start = self.index
        result = []
        
        while self.index < len(self.text):
            char = self.text[self.index]
            
            if char == '"':
                # End of string
                self.index += 1
                return ''.join(result)
            elif char == '\\':
                # Escape character
                self.index += 1
                if self.index >= len(self.text):
                    raise ValueError("Unexpected end of string")
                
                escape_char = self.text[self.index]
                if escape_char == '"':
                    result.append('"')
                elif escape_char == '\\':
                    result.append('\\')
                elif escape_char == '/':
                    result.append('/')
                elif escape_char == 'b':
                    result.append('\b')
                elif escape_char == 'f':
                    result.append('\f')
                elif escape_char == 'n':
                    result.append('\n')
                elif escape_char == 'r':
                    result.append('\r')
                elif escape_char == 't':
                    result.append('\t')
                elif escape_char == 'u':
                    # Unicode escape (simplified)
                    self.index += 1
                    unicode_hex = self.text[self.index:self.index+4]
                    result.append(chr(int(unicode_hex, 16)))
                    self.index += 3
                else:
                    raise ValueError(f"Invalid escape character: \\{escape_char}")
                self.index += 1
            else:
                result.append(char)
                self.index += 1
        
        raise ValueError("Unterminated string")
    
    def parse_number(self):
        """Parse number"""
        start = self.index
        
        # Negative sign
        if self.text[self.index] == '-':
            self.index += 1
        
        # Integer part
        if self.text[self.index] == '0':
            self.index += 1
        elif self.text[self.index].isdigit():
            while self.index < len(self.text) and self.text[self.index].isdigit():
                self.index += 1
        else:
            raise ValueError(f"Invalid number at position {start}")
        
        # Decimal part
        if self.index < len(self.text) and self.text[self.index] == '.':
            self.index += 1
            if not self.text[self.index].isdigit():
                raise ValueError(f"Invalid number at position {start}")
            while self.index < len(self.text) and self.text[self.index].isdigit():
                self.index += 1
        
        # Exponent part
        if self.index < len(self.text) and self.text[self.index] in 'eE':
            self.index += 1
            if self.text[self.index] in '+-':
                self.index += 1
            if not self.text[self.index].isdigit():
                raise ValueError(f"Invalid number at position {start}")
            while self.index < len(self.text) and self.text[self.index].isdigit():
                self.index += 1
        
        number_str = self.text[start:self.index]
        
        # Convert to int or float
        if '.' in number_str or 'e' in number_str or 'E' in number_str:
            return float(number_str)
        else:
            return int(number_str)
    
    def parse_true(self):
        """Parse true"""
        if self.text[self.index:self.index+4] == 'true':
            self.index += 4
            return True
        raise ValueError(f"Invalid value at position {self.index}")
    
    def parse_false(self):
        """Parse false"""
        if self.text[self.index:self.index+5] == 'false':
            self.index += 5
            return False
        raise ValueError(f"Invalid value at position {self.index}")
    
    def parse_null(self):
        """Parse null"""
        if self.text[self.index:self.index+4] == 'null':
            self.index += 4
            return None
        raise ValueError(f"Invalid value at position {self.index}")


# ==================== Collection Class ====================

class Collection:
    """
    NoSQL Collection class
    Similar to MongoDB Collection
    Stores JSON documents (usually an array of objects)
    """
    
    def __init__(self, documents=None):
        """
        Initialize Collection
        
        Parameters:
            documents: list - List of JSON documents [{}, {}, ...]
        """
        if documents is None:
            self.documents = []
        elif isinstance(documents, list):
            self.documents = documents
        else:
            raise ValueError("Documents must be a list")
    
    def __repr__(self):
        """Print Collection"""
        if not self.documents:
            return "Empty Collection"
        
        result = [f"Collection with {len(self.documents)} documents:"]
        result.append("-" * 60)
        
        # Display first 5 documents
        display_count = min(5, len(self.documents))
        for i in range(display_count):
            doc_str = str(self.documents[i])
            if len(doc_str) > 100:
                doc_str = doc_str[:97] + "..."
            result.append(f"[{i}] {doc_str}")
        
        if len(self.documents) > 5:
            result.append(f"... {len(self.documents) - 5} more documents")
        
        return "\n".join(result)
    
    def __len__(self):
        """Return the number of documents"""
        return len(self.documents)
    
    def __getitem__(self, index):
        """Support index access"""
        return self.documents[index]
    
    # ==================== 1. Find (Filtering) ====================
    
    def find(self, query=None):
        """
        Find documents (similar to MongoDB find)
        
        Parameters:
            query: dict or callable
                - dict: {'field': value} or {'field': {'$gt': value}}
                - callable: lambda doc: condition
        
        Returns:
            Collection: Matched documents
        
        Examples:
            collection.find({'Country': 'USA'})
            collection.find({'GNP': {'$gt': 1000000}})
            collection.find(lambda doc: doc['GNP'] > 1000000)
        """
        if query is None:
            # Return all documents
            return Collection(self.documents[:])
        
        if callable(query):
            # Use function filtering
            matched = [doc for doc in self.documents if query(doc)]
            return Collection(matched)
        
        elif isinstance(query, dict):
            # Use query dictionary
            matched = []
            for doc in self.documents:
                if self._match_query(doc, query):
                    matched.append(doc)
            return Collection(matched)
        
        else:
            raise TypeError("Query must be dict or callable")
    
    def _match_query(self, doc, query):
        """Check if document matches query"""
        for field, condition in query.items():
            # Get value from document (support nested fields)
            value = self._get_nested_value(doc, field)
            
            if isinstance(condition, dict):
                # Operator query: {'$gt': 100}
                for op, op_value in condition.items():
                    if op == '$gt':
                        if not (value is not None and value > op_value):
                            return False
                    elif op == '$gte':
                        if not (value is not None and value >= op_value):
                            return False
                    elif op == '$lt':
                        if not (value is not None and value < op_value):
                            return False
                    elif op == '$lte':
                        if not (value is not None and value <= op_value):
                            return False
                    elif op == '$eq':
                        if value != op_value:
                            return False
                    elif op == '$ne':
                        if value == op_value:
                            return False
                    elif op == '$in':
                        if value not in op_value:
                            return False
                    else:
                        raise ValueError(f"Unknown operator: {op}")
            else:
                # Simple equality query
                if value != condition:
                    return False
        
        return True
    
    def _get_nested_value(self, doc, field):
        """Get value of nested field (support 'field.subfield')"""
        if '.' in field:
            parts = field.split('.')
            value = doc
            for part in parts:
                if isinstance(value, dict) and part in value:
                    value = value[part]
                else:
                    return None
            return value
        else:
            return doc.get(field)
    
    # ==================== 2. Project (Projection) ====================
    
    def project(self, fields):
        """
        Projection - select specific fields
        
        Parameters:
            fields: dict or list
                - dict: {'field1': 1, 'field2': 1} (include) or {'field1': 0} (exclude)
                - list: ['field1', 'field2'] (include these fields)
        
        Returns:
            Collection: Projected documents
        
        Examples:
            collection.project({'Name': 1, 'GNP': 1})
            collection.project(['Name', 'GNP'])
        """
        if isinstance(fields, list):
            # List form: include specified fields
            projected = []
            for doc in self.documents:
                new_doc = {field: doc.get(field) for field in fields if field in doc}
                projected.append(new_doc)
            return Collection(projected)
        
        elif isinstance(fields, dict):
            # Dict form
            include_fields = [k for k, v in fields.items() if v == 1]
            exclude_fields = [k for k, v in fields.items() if v == 0]
            
            if include_fields and exclude_fields:
                raise ValueError("Cannot mix inclusion and exclusion in projection")
            
            projected = []
            for doc in self.documents:
                if include_fields:
                    # Inclusion mode
                    new_doc = {field: doc.get(field) for field in include_fields if field in doc}
                else:
                    # Exclusion mode
                    new_doc = {k: v for k, v in doc.items() if k not in exclude_fields}
                projected.append(new_doc)
            
            return Collection(projected)
        
        else:
            raise TypeError("Fields must be dict or list")
    
    # ==================== 3. Group & Aggregate ====================
    
    def group(self, group_by, aggregations):
        """
        Group and Aggregate
        
        Parameters:
            group_by: str or list - Grouping fields
            aggregations: dict - {output_field: {operator: input_field}}
        
        Returns:
            Collection: Aggregated results
        
        Examples:
            collection.group('Continent', {
                'max_gnp': {'$max': 'GNP'},
                'total_pop': {'$sum': 'Population'}
            })
        """
        if isinstance(group_by, str):
            group_by = [group_by]
        
        # Create groups
        groups = {}
        for doc in self.documents:
            # Create group key
            key_values = tuple(self._get_nested_value(doc, field) for field in group_by)
            
            if key_values not in groups:
                groups[key_values] = []
            groups[key_values].append(doc)
        
        # Aggregate for each group
        results = []
        for key_values, group_docs in groups.items():
            result_doc = {}
            
            # Add grouping fields
            for i, field in enumerate(group_by):
                result_doc[field] = key_values[i]
            
            # Perform aggregations
            for output_field, agg_spec in aggregations.items():
                for operator, input_field in agg_spec.items():
                    values = [self._get_nested_value(doc, input_field) for doc in group_docs]
                    values = [v for v in values if v is not None]
                    
                    if operator == '$sum':
                        result_doc[output_field] = sum(values) if values else 0
                    elif operator == '$avg':
                        result_doc[output_field] = sum(values) / len(values) if values else None
                    elif operator == '$max':
                        result_doc[output_field] = max(values) if values else None
                    elif operator == '$min':
                        result_doc[output_field] = min(values) if values else None
                    elif operator == '$count':
                        result_doc[output_field] = len(values)
                    else:
                        raise ValueError(f"Unknown aggregation operator: {operator}")
            
            results.append(result_doc)
        
        return Collection(results)
    
    # ==================== 4. Join ====================
    
    def join(self, other, left_field, right_field, how='inner'):
        """
        Join two Collections
        
        Parameters:
            other: Collection - The other Collection to join
            left_field: str - Left join key
            right_field: str - Right join key
            how: str - Join type ('inner', 'left', 'right', 'outer')
        
        Returns:
            Collection: Joined documents
        
        Examples:
            countries.join(languages, 'Code', 'CountryCode')
        """
        # Build right index
        right_index = {}
        for doc in other.documents:
            key = self._get_nested_value(doc, right_field)
            if key not in right_index:
                right_index[key] = []
            right_index[key].append(doc)
        
        results = []
        matched_right_keys = set()
        
        # Iterate over left documents
        for left_doc in self.documents:
            left_key = self._get_nested_value(left_doc, left_field)
            
            if left_key in right_index:
                # Found matches
                for right_doc in right_index[left_key]:
                    matched_right_keys.add(left_key)
                    # Merge documents
                    merged = {**left_doc, **right_doc}
                    results.append(merged)
            elif how in ['left', 'outer']:
                # Left join: keep left documents
                results.append(left_doc.copy())
        
        # Handle right join or full outer join
        if how in ['right', 'outer']:
            for right_doc in other.documents:
                right_key = self._get_nested_value(right_doc, right_field)
                if right_key not in matched_right_keys:
                    results.append(right_doc.copy())
        
        return Collection(results)
    
    # ==================== Helper Methods ====================
    
    def sort(self, field, ascending=True):
        """Sort documents"""
        sorted_docs = sorted(
            self.documents,
            key=lambda doc: self._get_nested_value(doc, field) or 0,
            reverse=not ascending
        )
        return Collection(sorted_docs)
    
    def limit(self, n):
        """Limit the number of returned documents"""
        return Collection(self.documents[:n])
    
    def count(self):
        """Return the number of documents"""
        return len(self.documents)
    
    def to_list(self):
        """Convert to list"""
        return self.documents[:]
    
    @classmethod
    def from_json(cls, filepath):
        """Create Collection from JSON file"""
        parser = JSONParser()
        data = parser.load(filepath)
        
        if data is None:
            raise ValueError(f"Failed to load JSON from {filepath}")
        
        if not isinstance(data, list):
            raise ValueError("JSON must be an array of documents")
        
        return cls(data)